# FBref Scraper — notebook de pruebas (una liga)

Notebook autocontenido para probar el scraper **celda a celda** contra una sola liga
(por defecto **La Liga**, `comp_id=12`).

**Reglas que respeta (y que no debes tocar):**
- Límite oficial de FBref: **10 peticiones/minuto**. Aquí: 1 petición cada 7–10 s.
- Cada página se **cachea en disco** y no se vuelve a pedir (salvo `force=True`).
- Un **403/429** lanza `FBrefJailError`: parar y reintentar en unas horas (el jail dura hasta 24 h).

Cada celda de red hace **como mucho 1 petición nueva** (o ninguna, si ya está en caché),
así puedes ir paso a paso sin miedo.

In [ ]:
# Dependencias (ejecutar una vez)
!py -m pip install -q curl_cffi beautifulsoup4 pandas lxml

In [ ]:
# ============================== CONFIG ==============================
# Una sola liga para las pruebas. Cambia estos dos valores cuando quieras
# probar otra (verifica el comp_id en la URL del navegador: /en/comps/<id>/).

COMP_ID = 12                 # La Liga
SEASON = "2025-2026"
CACHE_DIR = "data/raw/fbref/html"

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Cliente HTTP
Rate limiting + caché en disco + fingerprint TLS de Chrome (`curl_cffi`).

In [ ]:
import hashlib
import random
import time
from pathlib import Path
from urllib.parse import urlparse

from curl_cffi import requests as cffi

logger = logging.getLogger("fbref.client")

BASE_URL = "https://fbref.com"


class FBrefJailError(Exception):
    """FBref nos ha bloqueado temporalmente (403/429). No reintentar: esperar horas."""


class FBrefNotFound(Exception):
    """404: la pagina no existe (liga/temporada/tabla sin datos)."""


def _url_to_cache_path(cache_dir: Path, url: str) -> Path:
    parsed = urlparse(url)
    path = parsed.path.strip("/") or "index"
    digest = hashlib.sha1((parsed.path + "?" + parsed.query).encode()).hexdigest()[:8]
    parts = path.split("/")
    filename = f"{parts[-1] or 'index'}-{digest}.html"
    return cache_dir.joinpath(*parts[:-1], filename)


class FBrefClient:
    def __init__(self, cache_dir=CACHE_DIR, min_interval=7.0, jitter=3.0,
                 impersonate="chrome", timeout=30):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.min_interval = min_interval
        self.jitter = jitter
        self.timeout = timeout
        self._last_request_ts = 0.0
        self._session = cffi.Session(impersonate=impersonate)
        self._session.headers.update({
            "Accept-Language": "en-US,en;q=0.9,es;q=0.8",
            "Referer": BASE_URL + "/",
        })

    def cache_path(self, url: str) -> Path:
        return _url_to_cache_path(self.cache_dir, url)

    def is_cached(self, url: str) -> bool:
        return self.cache_path(url).exists()

    def get(self, url_or_path: str, *, force: bool = False, max_retries: int = 2) -> str:
        url = url_or_path if url_or_path.startswith("http") else BASE_URL + url_or_path
        cpath = self.cache_path(url)

        if cpath.exists() and not force:
            logger.info("cache hit: %s", url)
            return cpath.read_text(encoding="utf-8")

        html = self._fetch_with_backoff(url, max_retries=max_retries)
        cpath.parent.mkdir(parents=True, exist_ok=True)
        cpath.write_text(html, encoding="utf-8")
        logger.info("fetched + cached: %s -> %s", url, cpath)
        return html

    def _fetch_with_backoff(self, url: str, max_retries: int) -> str:
        attempt = 0
        while True:
            self._respect_rate_limit()
            resp = self._session.get(url, timeout=self.timeout, allow_redirects=True)
            self._last_request_ts = time.monotonic()

            if resp.status_code == 200:
                return resp.text
            if resp.status_code == 404:
                raise FBrefNotFound(url)
            if resp.status_code in (403, 429):
                retry_after = resp.headers.get("Retry-After")
                # Guardar el cuerpo del bloqueo para diagnosticar la causa
                blocked_dir = self.cache_dir / "_blocked"
                blocked_dir.mkdir(parents=True, exist_ok=True)
                body_path = blocked_dir / f"{int(time.time())}-{resp.status_code}.html"
                body_path.write_text(resp.text or "", encoding="utf-8")
                raise FBrefJailError(
                    f"HTTP {resp.status_code} en {url}. Retry-After={retry_after!r}. "
                    f"Cuerpo guardado en {body_path}. "
                    "Detener el scraping; ejecuta la celda de diagnostico."
                )
            attempt += 1
            if attempt > max_retries:
                resp.raise_for_status()
            wait = 15 * (2 ** (attempt - 1)) + random.uniform(0, 5)
            logger.warning("HTTP %s en %s, reintento en %.0fs", resp.status_code, url, wait)
            time.sleep(wait)

    def _respect_rate_limit(self):
        elapsed = time.monotonic() - self._last_request_ts
        wait = self.min_interval + random.uniform(0, self.jitter) - elapsed
        if wait > 0:
            logger.info("rate limit: esperando %.1fs", wait)
            time.sleep(wait)

## 2. Parser
FBref esconde muchas tablas dentro de **comentarios HTML** — se descomentan antes de buscar.
También: aplanado de MultiIndex, limpieza de cabeceras repetidas y extracción de enlaces por fila
(de ahí salen los ids estables de jugador/equipo/partido).

In [ ]:
from io import StringIO

import pandas as pd
from bs4 import BeautifulSoup, Comment


def _soup_with_uncommented_tables(html: str) -> BeautifulSoup:
    soup = BeautifulSoup(html, "lxml")
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
        if "<table" in comment:
            comment.replace_with(BeautifulSoup(comment, "lxml"))
    return soup


def list_table_ids(html: str) -> list:
    soup = _soup_with_uncommented_tables(html)
    return [t.get("id") for t in soup.find_all("table") if t.get("id")]


def table_by_id(html: str, table_id: str):
    soup = _soup_with_uncommented_tables(html)
    table = soup.find("table", id=table_id)
    return None if table is None else _to_dataframe(table)


def table_by_id_prefix(html: str, prefix: str):
    soup = _soup_with_uncommented_tables(html)
    table = soup.find("table", id=lambda i: i and i.startswith(prefix))
    return None if table is None else _to_dataframe(table)


def _to_dataframe(table) -> pd.DataFrame:
    df = pd.read_html(StringIO(str(table)))[0]
    df = _flatten_columns(df)
    df = _drop_repeated_headers(df)
    links = _extract_row_links(table)
    if links and len(links) == len(df):
        df["_links"] = links
    return df


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        cols = []
        for top, bottom in df.columns:
            top = "" if str(top).startswith("Unnamed") else top
            cols.append(f"{top} {bottom}".strip())
        df.columns = cols
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df


def _drop_repeated_headers(df: pd.DataFrame) -> pd.DataFrame:
    first_col = df.columns[0]
    values_norm = df[first_col].astype(str).str.strip().str.lower().str.replace(" ", "_")
    df = df[values_norm != first_col]
    return df.dropna(how="all").reset_index(drop=True)


def _extract_row_links(table) -> list:
    tbody = table.find("tbody")
    if tbody is None:
        return []
    rows = []
    for tr in tbody.find_all("tr"):
        if tr.get("class") and ("thead" in tr["class"] or "spacer" in tr["class"]):
            continue
        cells = tr.find_all(["th", "td"])
        if all(not c.get_text(strip=True) for c in cells):
            continue
        row_links = {}
        for cell in cells:
            a = cell.find("a")
            if a and a.get("href"):
                row_links[cell.get("data-stat", "unknown")] = a["href"]
        rows.append(row_links)
    return rows

### 2.1 Test offline del parser (sin tocar la red)
Fixture con las rarezas típicas de FBref. Debe salir: 2 tablas detectadas,
2 filas de jugadores (sin la cabecera repetida) y `_links` alineados.

In [ ]:
_fixture = """
<html><body>
<table id="results2025-2026_overall">
<thead><tr><th>Rk</th><th>Squad</th><th>Pts</th></tr></thead>
<tbody>
<tr><th data-stat="rank">1</th><td data-stat="team"><a href="/en/squads/abc/Barca">Barcelona</a></td><td>88</td></tr>
</tbody></table>
<!--
<table id="stats_standard">
<thead>
<tr><th></th><th></th><th colspan="2">Per 90 Minutes</th></tr>
<tr><th>Player</th><th>Squad</th><th>Gls</th><th>Ast</th></tr>
</thead>
<tbody>
<tr><th data-stat="player"><a href="/en/players/xyz/Lewa">Lewandowski</a></th><td>Barcelona</td><td>0.85</td><td>0.2</td></tr>
<tr class="thead"><th>Player</th><td>Squad</td><td>Gls</td><td>Ast</td></tr>
<tr><th data-stat="player"><a href="/en/players/uvw/Mbappe">Mbappe</a></th><td>Real Madrid</td><td>0.9</td><td>0.3</td></tr>
</tbody></table>
-->
</body></html>
"""

assert list_table_ids(_fixture) == ["results2025-2026_overall", "stats_standard"]
_df = table_by_id(_fixture, "stats_standard")
assert len(_df) == 2, _df
assert _df["_links"].iloc[0] == {"player": "/en/players/xyz/Lewa"}
print("Parser OK")
_df

Parser OK


,player,squad,per_90_minutes_gls,per_90_minutes_ast,_links
0,Lewandowski,Barcelona,0.85,0.2,{'player': '/en/players/xyz/Lewa'}
1,Mbappe,Real Madrid,0.9,0.3,{'player': '/en/players/uvw/Mbappe'}


## 3. API de alto nivel

In [ ]:
# Ids de competicion de FBref (numero en /en/comps/<id>/). Verifica en el
# navegador antes de anadir uno nuevo.
COMPETITIONS = {
    "ENG-Premier League": 9,
    "ESP-La Liga": 12,
    "ITA-Serie A": 11,
    "GER-Bundesliga": 20,
    "FRA-Ligue 1": 13,
    "ENG-Championship": 10,
    "ESP-Segunda Division": 17,
    "POR-Primeira Liga": 32,
    "NED-Eredivisie": 23,
    "UEFA-Champions League": 8,
    "UEFA-Europa League": 19,
}

STAT_PAGES = {
    "standard": "stats",
    "keepers": "keepers",
    "keepers_adv": "keepersadv",
    "shooting": "shooting",
    "passing": "passing",
    "passing_types": "passing_types",
    "goal_shot_creation": "gca",
    "defense": "defense",
    "possession": "possession",
    "playing_time": "playingtime",
    "misc": "misc",
}

_TABLE_KEYS = {
    "standard": "standard",
    "keepers": "keeper",
    "keepers_adv": "keeper_adv",
    "shooting": "shooting",
    "passing": "passing",
    "passing_types": "passing_types",
    "goal_shot_creation": "gca",
    "defense": "defense",
    "possession": "possession",
    "playing_time": "playing_time",
    "misc": "misc",
}

api_logger = logging.getLogger("fbref.api")


class FBref:
    def __init__(self, cache_dir=CACHE_DIR, **client_kwargs):
        self.client = FBrefClient(cache_dir=cache_dir, **client_kwargs)

    def _season_page(self, comp_id, season, page, force):
        url = f"/en/comps/{comp_id}/{season}/{page}/" if page else f"/en/comps/{comp_id}/{season}/"
        try:
            return self.client.get(url, force=force)
        except FBrefNotFound:
            api_logger.info("pagina no disponible: %s", url)
            return None

    def available_tables(self, comp_id, season, stat="standard"):
        html = self._season_page(comp_id, season, STAT_PAGES[stat], force=False)
        return list_table_ids(html) if html else []

    def schedule(self, comp_id, season, force=False):
        html = self._season_page(comp_id, season, "schedule", force=force)
        if html is None:
            return None
        df = table_by_id_prefix(html, "sched")
        if df is not None and "date" in df.columns:
            df = df[df["date"].notna()].reset_index(drop=True)
        return df

    def team_stats(self, comp_id, season, stat="standard", against=False, force=False):
        html = self._season_page(comp_id, season, STAT_PAGES[stat], force=force)
        if html is None:
            return None
        suffix = "against" if against else "for"
        table_id = f"stats_squads_{_TABLE_KEYS[stat]}_{suffix}"
        df = table_by_id(html, table_id)
        if df is None:
            api_logger.info("tabla %s ausente en comp %s %s", table_id, comp_id, season)
        return df

    def player_stats(self, comp_id, season, stat="standard", force=False):
        html = self._season_page(comp_id, season, STAT_PAGES[stat], force=force)
        if html is None:
            return None
        df = table_by_id(html, f"stats_{_TABLE_KEYS[stat]}")
        if df is None:
            api_logger.info("tabla de jugadores %s ausente en comp %s %s", stat, comp_id, season)
        return df

    def league_table(self, comp_id, season, force=False):
        html = self._season_page(comp_id, season, "", force=force)
        if html is None:
            return None
        return table_by_id_prefix(html, "results")

    def match_report(self, match_href, force=False):
        try:
            return self.client.get(match_href, force=force)
        except FBrefNotFound:
            return None


fb = FBref()
print(f"Liga de pruebas: comp_id={COMP_ID}, season={SEASON}")

Liga de pruebas: comp_id=12, season=2025-2026


## 4. Primera petición real: ¿qué tablas ofrece esta liga?
**1 petición de red** (la página `stats`). Si aquí salta `FBrefJailError`,
tu IP ya estaba en jail antes de empezar — espera unas horas.

In [ ]:
def donde_guardar(url_or_path):
    """Dada una URL de FBref, dice en que fichero guardar el HTML
    para que la cache lo encuentre. Guarda con: Ctrl+S -> 'Pagina web, solo HTML'."""
    url = url_or_path if url_or_path.startswith("http") else BASE_URL + url_or_path
    destino = _url_to_cache_path(Path(CACHE_DIR), url)
    destino.parent.mkdir(parents=True, exist_ok=True)
    print("1. Abre en tu navegador:")
    print("  ", url)
    print("2. Ctrl+S -> tipo 'Pagina web, solo HTML' -> guarda como:")
    print("  ", destino)
    return destino

# La pagina de stats que necesita el paso 4:
donde_guardar(f"/en/comps/{COMP_ID}/{SEASON}/stats/")

1. Abre en tu navegador:
   https://fbref.com/en/comps/12/2025-2026/stats/
2. Ctrl+S -> tipo 'Pagina web, solo HTML' -> guarda como:
   data\raw\fbref\html\en\comps\12\2025-2026\stats-bae61427.html


WindowsPath('data/raw/fbref/html/en/comps/12/2025-2026/stats-bae61427.html')

In [ ]:
fb = FBref()
donde_guardar(f"/en/comps/{COMP_ID}/{SEASON}/stats/")

1. Abre en tu navegador:
   https://fbref.com/en/comps/12/2025-2026/stats/
2. Ctrl+S -> tipo 'Pagina web, solo HTML' -> guarda como:
   data\raw\fbref\html\en\comps\12\2025-2026\stats-bae61427.html


WindowsPath('data/raw/fbref/html/en/comps/12/2025-2026/stats-bae61427.html')

In [ ]:
ids = fb.available_tables(COMP_ID, SEASON)
print(f"{len(ids)} tablas en la pagina de stats:")
for i in ids:
    print(" -", i)

INFO fbref.client: cache hit: https://fbref.com/en/comps/12/2025-2026/stats/


3 tablas en la pagina de stats:
 - stats_squads_standard_for
 - stats_squads_standard_against
 - stats_standard


In [ ]:
donde_guardar(f"/en/comps/{COMP_ID}/{SEASON}/schedule/")

1. Abre en tu navegador:
   https://fbref.com/en/comps/12/2025-2026/schedule/
2. Ctrl+S -> tipo 'Pagina web, solo HTML' -> guarda como:
   data\raw\fbref\html\en\comps\12\2025-2026\schedule-220617e4.html


WindowsPath('data/raw/fbref/html/en/comps/12/2025-2026/schedule-220617e4.html')

In [ ]:
sched = fb.schedule(COMP_ID, SEASON)
print(sched.shape)
sched.head(10)

INFO fbref.client: cache hit: https://fbref.com/en/comps/12/2025-2026/schedule/


(380, 13)


,wk,day,date,time,home,score,away,attendance,venue,referee,match_report,notes,_links
0,1.0,Fri,2025-08-15,19:00,Girona,1–3,Rayo Vallecano,12403.0,Estadi Municipal de Montilivi,Javier Alberola,Match Report,NaN,"{'date': '/en/matches/2025-08-15', 'home_team'..."
1,1.0,Fri,2025-08-15,21:30,Villarreal,2–0,Oviedo,18333.0,Estadio de la Cerámica,Alejandro Muñíz,Match Report,NaN,"{'date': '/en/matches/2025-08-15', 'home_team'..."
2,1.0,Sat,2025-08-16,19:30,Mallorca,0–3,Barcelona,23318.0,Estadi Mallorca Son Moix,José Luis Munuera,Match Report,NaN,"{'date': '/en/matches/2025-08-16', 'home_team'..."
3,1.0,Sat,2025-08-16,21:30,Valencia,1–1,Real Sociedad,45333.0,Estadio de Mestalla,Jose Maria Sánchez,Match Report,NaN,"{'date': '/en/matches/2025-08-16', 'home_team'..."
4,1.0,Sat,2025-08-16,21:30,Alavés,2–1,Levante,12837.0,Estadio de Mendizorroza,Miguel Sesma,Match Report,NaN,"{'date': '/en/matches/2025-08-16', 'home_team'..."
5,1.0,Sun,2025-08-17,17:00,Celta Vigo,0–2,Getafe,20530.0,Estadio Abanca Balaídos,Iosu Galech,Match Report,NaN,"{'date': '/en/matches/2025-08-17', 'home_team'..."
6,1.0,Sun,2025-08-17,19:30,Athletic Club,3–2,Sevilla,49134.0,San Mamés,Francisco Hernández,Match Report,NaN,"{'date': '/en/matches/2025-08-17', 'home_team'..."
7,1.0,Sun,2025-08-17,21:30,Espanyol,2–1,Atlético Madrid,29612.0,RCDE Stadium,Mateo Busquets,Match Report,NaN,"{'date': '/en/matches/2025-08-17', 'home_team'..."
8,1.0,Mon,2025-08-18,21:00,Elche,1–1,Real Betis,28585.0,Estadio Manuel Martínez Valero,Víctor García,Match Report,NaN,"{'date': '/en/matches/2025-08-18', 'home_team'..."
9,1.0,Tue,2025-08-19,21:00,Real Madrid,1–0,Osasuna,68407.0,Estadio Santiago Bernabéu,Adrián Cordero,Match Report,NaN,"{'date': '/en/matches/2025-08-19', 'home_team'..."


In [ ]:
players = fb.player_stats(COMP_ID, SEASON, "standard")
print(players.shape)
players.head()

INFO fbref.client: cache hit: https://fbref.com/en/comps/12/2025-2026/stats/


(611, 26)


,rk,player,nation,pos,squad,age,born,playing_time_mp,playing_time_starts,playing_time_min,...,performance_pkatt,performance_crdy,performance_crdr,per_90_minutes_gls,per_90_minutes_ast,per_90_minutes_g+a,per_90_minutes_g-pk,per_90_minutes_g+a-pk,matches,_links
0,1,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,19,2006,22,4,681,...,0,0,0,0.26,0.00,0.26,0.26,0.26,Matches,{'player': '/en/players/29dd3b2e/Jones-El-Abde...
1,2,Tay Abed,il ISR,MF,Levante,21,2004,7,0,93,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,"{'player': '/en/players/88824111/Tay-Abed', 'n..."
2,3,Abdel Abqar,ma MAR,DF,Getafe,26,1999,23,17,1437,...,0,10,1,0.00,0.13,0.13,0.00,0.13,Matches,"{'player': '/en/players/2bc7f211/Abdel-Abqar',..."
3,4,Akor Adams,ng NGA,FW,Sevilla,25,2000,32,21,2064,...,3,3,0,0.44,0.13,0.57,0.31,0.44,Matches,"{'player': '/en/players/2ccd39ed/Akor-Adams', ..."
4,5,Adrián,es ESP,GK,Real Betis,38,1987,1,1,90,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,"{'player': '/en/players/f76e6b4e/Adrian', 'nat..."


In [ ]:
!py -m pip install -q browser-cookie3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
import browser_cookie3

class FBrefClientCookies(FBrefClient):
    """FBrefClient que inyecta las cookies de tu Chrome (incluida cf_clearance).
    Requiere haber visitado fbref.com en Chrome antes."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._load_browser_cookies()

    def _load_browser_cookies(self):
        cj = browser_cookie3.chrome(domain_name="fbref.com")
        n = 0
        for c in cj:
            self._session.cookies.set(c.name, c.value, domain=c.domain)
            n += 1
        names = [c.name for c in cj]
        print(f"{n} cookies cargadas de Chrome: {names}")
        if not any("cf_clearance" in x for x in names):
            print("AVISO: no aparece cf_clearance. Visita fbref.com en Chrome y reintenta.")


# Cambiar el cliente por la version con cookies
fb.client = FBrefClientCookies()
print("fb ahora usa curl_cffi + cookies de Chrome. Prueba una pagina NO cacheada.")

BrowserCookieError: Unable to get key for cookie decryption

In [ ]:
COOKIES_MANUALES = {
    "cf_clearance": "pgqrQrNqqfXjmDdeoFUksPvFwHWLdIxu2JAGkQ4kFMg",
}

USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"

class FBrefClientManual(FBrefClient):
    """Cookie + User-Agent copiados a mano del navegador, alineados."""
    def __init__(self, cookies, user_agent, *args, **kwargs):
        super().__init__(*args, **kwargs)
        for k, v in cookies.items():
            self._session.cookies.set(k, v, domain=".fbref.com")
        self._session.headers["User-Agent"] = user_agent
        print(f"cookie + UA cargados. UA = {user_agent[:50]}...")

fb.client = FBrefClientManual(COOKIES_MANUALES, USER_AGENT)

cookie + UA cargados. UA = Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb...


In [ ]:
shooting = fb.team_stats(COMP_ID, SEASON, "shooting", force=True)
shooting.head() if shooting is not None else print("sin datos de shooting")

FBrefJailError: HTTP 403 en https://fbref.com/en/comps/12/2025-2026/shooting/. Retry-After=None. Cuerpo guardado en data\raw\fbref\html\_blocked\1783505031-403.html. Detener el scraping; ejecuta la celda de diagnostico.

## 4-bis. Diagnóstico si salta `FBrefJailError`
Ejecuta esta celda **sin volver a pedir nada**: lee el cuerpo del último 403 guardado
y te dice qué tipo de bloqueo es y qué hacer. Cada tipo tiene remedio distinto —
reintentar a ciegas solo alarga el castigo.

In [ ]:
from pathlib import Path

blocked = sorted(Path(CACHE_DIR, "_blocked").glob("*.html"))
if not blocked:
    print("No hay bloqueos registrados todavia.")
else:
    body = blocked[-1].read_text(encoding="utf-8").lower()
    print(f"Analizando {blocked[-1].name} ({len(body)} bytes)\n")
    if "rate limited" in body or "bot-traffic" in body or "sports reference" in body:
        print("-> JAIL de Sports Reference: superaste las 10 req/min en algun momento")
        print("   (posiblemente en tus intentos anteriores). Unica solucion: esperar,")
        print("   hasta 24h. Ningun cambio de codigo lo arregla.")
    elif "just a moment" in body or "challenge" in body or "cf-chl" in body or "turnstile" in body:
        print("-> CHALLENGE JS de Cloudflare: el cliente HTTP no pasa el filtro en tu red.")
        print("   Solucion: usa el fallback de Playwright (seccion 4-ter). Un navegador")
        print("   real pasa el challenge de forma legitima.")
    elif "1020" in body or "access denied" in body or "error code" in body:
        print("-> BLOQUEO por reputacion de IP (VPN / red corporativa / rango flaggeado).")
        print("   Solucion: cambia de red (casa, hotspot movil) y prueba de nuevo.")
    else:
        print("-> Tipo no reconocido. Inspecciona el HTML a mano:")
        print(f"   {blocked[-1]}")
        print("\nPrimeros 500 chars:\n", body[:500])

Analizando 1783503947-403.html (6081 bytes)

-> CHALLENGE JS de Cloudflare: el cliente HTTP no pasa el filtro en tu red.
   Solucion: usa el fallback de Playwright (seccion 4-ter). Un navegador
   real pasa el challenge de forma legitima.


## 4-ter. Fallback: Playwright (navegador real)
Solo si el diagnóstico dice **challenge de Cloudflare** y `fbref.com` te abre bien en tu navegador.

- Mismo rate limit (7–10 s) y **misma caché** → el resto del notebook funciona sin cambios:
  simplemente haz `fb.client = pw` y sigue con el paso 4.
- Se lanza **con ventana visible** (`headless=False`): si Cloudflare te muestra un challenge
  interactivo, **resuélvelo tú a mano en la ventana** — eso eres tú demostrando que eres humano,
  y es donde está el límite: no vamos a automatizar la resolución de challenges/CAPTCHAs.
- Instala una vez con la celda siguiente (en Windows: `py -m playwright install chromium`).

Nota técnica (Windows + Jupyter): el loop del kernel no soporta subprocesos, así que
Playwright corre en un hilo aparte con su propio loop Proactor. El uso desde el notebook
es 100% síncrono: `pw.start()`, `pw.get(...)`, `pw.stop()` — sin `await`.

In [ ]:
!py -m pip install -q playwright
!py -m playwright install chromium

In [ ]:
import asyncio
import random
import sys
import threading
import time

from playwright.async_api import async_playwright


class PlaywrightFetcher:
    """Chromium real en un hilo con su propio event loop Proactor.

    Necesario en Windows + Jupyter: el loop del kernel es de tipo selector
    y no puede lanzar subprocesos. Aqui todo Playwright vive en su hilo y
    el notebook lo usa de forma sincrona: pw.start(), pw.get(...), pw.stop().
    Mismo contrato y misma cache que FBrefClient -> fb.client = pw.
    """

    def __init__(self, cache_dir=CACHE_DIR, min_interval=7.0, jitter=3.0, headless=False):
        self.cache_dir = Path(cache_dir)
        self.min_interval = min_interval
        self.jitter = jitter
        self.headless = headless
        self._last_request_ts = 0.0
        self._loop = None
        self._thread = None
        self._pw = None
        self._context = None
        self._page = None

    # ------------------------------------------------- hilo + event loop

    def start(self):
        if sys.platform == "win32":
            self._loop = asyncio.ProactorEventLoop()
        else:
            self._loop = asyncio.new_event_loop()

        def _run():
            asyncio.set_event_loop(self._loop)
            self._loop.run_forever()

        self._thread = threading.Thread(target=_run, daemon=True, name="playwright-loop")
        self._thread.start()
        self._call(self._astart())
        print("Navegador listo. Si ves un challenge en la ventana, resuelvelo a mano.")

    def stop(self):
        if self._loop is None:
            return
        self._call(self._astop())
        self._loop.call_soon_threadsafe(self._loop.stop)
        self._thread.join(timeout=10)
        self._loop = None

    def _call(self, coro, timeout=300):
        fut = asyncio.run_coroutine_threadsafe(coro, self._loop)
        return fut.result(timeout=timeout)

    # ------------------------------------------------- parte async interna

    async def _astart(self):
        self._pw = await async_playwright().start()
        profile_dir = r"C:\Users\daniel.cejas\AppData\Local\Google\Chrome\User Data"
        self._context = await self._pw.chromium.launch_persistent_context(
            user_data_dir=profile_dir,
            channel="chrome",
            headless=False,
            locale="en-US",
            args=["--disable-blink-features=AutomationControlled"],
        )
        self._page = self._context.pages[0] if self._context.pages else await self._context.new_page()
        await self._page.goto(BASE_URL + "/", wait_until="domcontentloaded")
        self._last_request_ts = time.monotonic()

    async def _astop(self):
        if self._context:
            await self._context.close()
        if self._pw:
            await self._pw.stop()

    async def _aget(self, url_or_path, force=False):
        url = url_or_path if url_or_path.startswith("http") else BASE_URL + url_or_path
        cpath = self.cache_path(url)
        if cpath.exists() and not force:
            logger.info("cache hit: %s", url)
            return cpath.read_text(encoding="utf-8")

        elapsed = time.monotonic() - self._last_request_ts
        wait = self.min_interval + random.uniform(0, self.jitter) - elapsed
        if wait > 0:
            await asyncio.sleep(wait)

        resp = await self._page.goto(url, wait_until="domcontentloaded")
        self._last_request_ts = time.monotonic()

        if resp and resp.status == 404:
            raise FBrefNotFound(url)
        if resp and resp.status in (403, 429):
            print("Bloqueo/challenge detectado. Si hay challenge en la ventana,")
            print("resuelvelo ahora: espero 60s y recargo...")
            await asyncio.sleep(60)
            resp = await self._page.goto(url, wait_until="domcontentloaded")
            self._last_request_ts = time.monotonic()
            if resp and resp.status in (403, 429):
                raise FBrefJailError(
                    f"HTTP {resp.status} en {url} tambien con navegador real. "
                    "Probable jail o bloqueo de IP: espera unas horas o cambia de red."
                )

        html = await self._page.content()
        cpath.parent.mkdir(parents=True, exist_ok=True)
        cpath.write_text(html, encoding="utf-8")
        logger.info("fetched + cached (playwright): %s", url)
        return html

    # ------------------------------------------------- API sincrona

    def cache_path(self, url):
        return _url_to_cache_path(self.cache_dir, url)

    def get(self, url_or_path, *, force=False, max_retries=2):
        return self._call(self._aget(url_or_path, force=force))


pw = PlaywrightFetcher(headless=False)
pw.start()

# Intercambiar el cliente y repetir el paso 4:
fb.client = pw
print("fb ahora usa Playwright. Vuelve a ejecutar la celda del paso 4.")

TargetClosedError: BrowserType.launch_persistent_context: Target page, context or browser has been closed
Browser logs:

<launching> C:\Program Files\Google\Chrome\Application\chrome.exe --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-edgeupdater --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints,msForceBrowserSignIn,msEdgeUpdateLaunchServicesPreferredVersion --enable-features

In [ ]:
try:
    pw.stop()
except Exception:
    pass
pw = PlaywrightFetcher()
pw.start()
fb.client = pw

Navegador listo. Si ves un challenge en la ventana, resuelvelo a mano.


## 5. Calendario y resultados
Reutiliza caché si ya la tiene; `force=True` solo cuando quieras refrescar la jornada.

In [ ]:
sched = fb.schedule(COMP_ID, SEASON)   # force=True para refrescar
print(sched.shape if sched is not None else "sin calendario")
sched.head(10)

In [ ]:
# Partidos ya jugados con match report disponible en _links
played = sched[sched["score"].notna() & (sched["score"].astype(str).str.strip() != "")]
reports = [l.get("match_report") for l in played["_links"] if isinstance(l, dict) and l.get("match_report")]
print(f"{len(played)} partidos jugados, {len(reports)} con match report")
reports[:3]

## 6. Stats de equipos (squad, 'for' y 'against')
La página `stats` ya está en caché desde el paso 4 → **0 peticiones nuevas**.

In [ ]:
squads_for = fb.team_stats(COMP_ID, SEASON, "standard")
squads_for.head()

In [ ]:
squads_against = fb.team_stats(COMP_ID, SEASON, "standard", against=True)
squads_against.head()

## 7. Stats de jugadores
Misma página en caché → **0 peticiones nuevas**. Aquí van los ids de jugador (en `_links`).

In [ ]:
players = fb.player_stats(COMP_ID, SEASON, "standard")
print(players.shape)
players.head()

In [ ]:
# Extraer el id estable de FBref del href: /en/players/<id>/<slug>
def fbref_id(href):
    parts = href.split("/")
    return parts[3] if len(parts) > 3 else None

players["player_id"] = players["_links"].map(
    lambda l: fbref_id(l["player"]) if isinstance(l, dict) and "player" in l else None
)
players[["player", "squad", "player_id"]].head()

## 8. Una página de stats avanzadas (shooting)
**1 petición nueva.** En ligas sin datos Opta devolverá `None` — comportamiento esperado, no error.

In [ ]:
shooting = fb.team_stats(COMP_ID, SEASON, "shooting")
shooting.head() if shooting is not None else print("Sin stats de shooting para esta liga (normal fuera de Big 5 / Opta)")

## 9. Un match report
**1 petición nueva** (y queda cacheado para siempre: los partidos jugados no cambian).
Los informes tienen tablas con ids dinámicos por equipo, así que primero se listan.

In [ ]:
if reports:
    html = fb.match_report(reports[0])
    ids_report = list_table_ids(html)
    print(f"{len(ids_report)} tablas en el informe:")
    for i in ids_report:
        print(" -", i)
else:
    print("No hay match reports todavia")

In [ ]:
# Ejemplo: cargar la primera tabla 'stats_..._summary' del informe
if reports:
    summary_id = next((i for i in ids_report if i and i.startswith("stats_") and i.endswith("_summary")), None)
    if summary_id:
        report_df = table_by_id(html, summary_id)
        display(report_df.head())
    else:
        print("Sin tabla summary (liga sin datos avanzados); prueba con otros ids de la lista")

## 10. Verificar la caché
Todo lo pedido queda como HTML crudo en disco — esta es tu capa raw.
Repetir cualquier celda anterior ya no toca la red.

In [ ]:
from pathlib import Path
files = sorted(Path(CACHE_DIR).rglob("*.html"))
print(f"{len(files)} paginas cacheadas:")
for f in files:
    print(f" - {f}  ({f.stat().st_size/1024:.0f} KB)")

## Próximos pasos
1. Repetir el paso 4 con otra liga (`COMP_ID = 17`, Segunda) y comparar qué tablas ofrece.
2. Decidir mapeo de columnas → `raw.fbref_*` en PostgreSQL (guarda también el HTML, ya lo tienes en disco).
3. Mover esto al paquete `fbref_scraper/` (ya lo tienes) y llamarlo desde el orquestador semanal.

**Recordatorio:** si salta `FBrefJailError`, cierra el notebook y vuelve en unas horas.
No bajes `min_interval` de 7 s.

In [ ]:
!py -m jupyter kernelspec list

Available kernels:
  ir         C:\Users\daniel.cejas\AppData\Roaming\jupyter\kernels\ir
  python3    C:\Users\daniel.cejas\AppData\Roaming\Python\share\jupyter\kernels\python3


In [ ]:
!py -m pip install jupyter_client

In [2]:
options(repos = c(CRAN = "https://cloud.r-project.org"))
install.packages("worldfootballR")
install.packages("readr")
install.packages("dplyr")

Installing package into 'C:/Users/daniel.cejas/AppData/Local/R/win-library/4.6'
(as 'lib' is unspecified)

also installing the dependencies 'sys', 'bit', 'askpass', 'bit64', 'generics', 'R6', 'mime', 'openssl', 'hms', 'snakecase', 'timechange', 'prettyunits', 'clipr', 'vroom', 'cpp11', 'tzdb', 'selectr', 'pkgconfig', 'dplyr', 'httr', 'janitor', 'lubridate', 'magrittr', 'progress', 'purrr', 'qdapRegex', 'readr', 'rstudioapi', 'rvest', 'stringi', 'stringr', 'tidyr', 'tidyselect', 'withr', 'xml2', 'tibble', 'curl'




package 'sys' successfully unpacked and MD5 sums checked
package 'bit' successfully unpacked and MD5 sums checked
package 'askpass' successfully unpacked and MD5 sums checked
package 'bit64' successfully unpacked and MD5 sums checked
package 'generics' successfully unpacked and MD5 sums checked
package 'R6' successfully unpacked and MD5 sums checked
package 'mime' successfully unpacked and MD5 sums checked
package 'openssl' successfully unpacked and MD5 sums checked
package 'hms' successfully unpacked and MD5 sums checked
package 'snakecase' successfully unpacked and MD5 sums checked
package 'timechange' successfully unpacked and MD5 sums checked
package 'prettyunits' successfully unpacked and MD5 sums checked
package 'clipr' successfully unpacked and MD5 sums checked
package 'vroom' successfully unpacked and MD5 sums checked
package 'cpp11' successfully unpacked and MD5 sums checked
package 'tzdb' successfully unpacked and MD5 sums checked
package 'selectr' successfully unpacked and M

Installing package into 'C:/Users/daniel.cejas/AppData/Local/R/win-library/4.6'
(as 'lib' is unspecified)



package 'readr' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\daniel.cejas\AppData\Local\Temp\Rtmpw5P85L\downloaded_packages


Installing package into 'C:/Users/daniel.cejas/AppData/Local/R/win-library/4.6'
(as 'lib' is unspecified)



package 'dplyr' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\daniel.cejas\AppData\Local\Temp\Rtmpw5P85L\downloaded_packages


In [3]:
library(worldfootballR)
library(readr)
library(dplyr)


Adjuntando el paquete: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




In [4]:
COUNTRY     <- "ESP"      # ESP, ENG, ITA, GER, FRA...
GENDER      <- "M"
SEASON_END  <- 2026       # 2025-2026 -> año final
TIER        <- "1st"      # "1st", "2nd"...
OUT_DIR     <- "data/raw/fbref_r"

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

save_csv <- function(df, name) {
  if (is.null(df) || nrow(df) == 0) {
    message(sprintf("  [vacio] %s -> no se guarda (liga sin esos datos?)", name))
    return(invisible(NULL))
  }
  path <- file.path(OUT_DIR, sprintf("%s_%s_%d.csv", name, COUNTRY, SEASON_END))
  write_csv(df, path)
  message(sprintf("  [ok] %s: %d filas -> %s", name, nrow(df), path))
}

In [5]:
message("BLOQUE 1: calendario y resultados...")

fixtures <- fb_match_results(
  country = COUNTRY, gender = GENDER,
  season_end_year = SEASON_END, tier = TIER
)
save_csv(fixtures, "match_results")

match_urls <- fixtures$MatchURL[!is.na(fixtures$MatchURL)]
message(sprintf("  %d partidos con URL de informe", length(match_urls)))

head(fixtures)

BLOQUE 1: calendario y resultados...

Warning message in session_set_response(x, resp):
"Forbidden (HTTP 403)."


ERROR: [1m[33mError[39m in `map()`:[22m
[1m[22m[36mℹ[39m In index: 1.
[1mCaused by error in `read_html.response()`:[22m
[33m![39m Forbidden (HTTP 403).
